In [ ]:
# Shared project setup for imports and file locations
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DATA_DIR = PROJECT_ROOT / 'data'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
FIGURES_DIR = PROJECT_ROOT / 'figures'

def resolve_path(path):
    candidate = Path(path)
    if candidate.exists():
        return candidate
    text = str(path).replace('\\', '/')
    name = Path(text).name
    special = {
        ARTIFACTS_DIR / 'controls' / 'positive_controls.pkl': ARTIFACTS_DIR / 'controls' / 'positive_controls.pkl',
        ARTIFACTS_DIR / 'controls' / 'negative_controls.pkl': ARTIFACTS_DIR / 'controls' / 'negative_controls.pkl',
        'Ten_positive_controls_1119.pkl': ARTIFACTS_DIR / 'controls' / 'positive_controls.pkl',
        'Ten_negative_controls_1119.pkl': ARTIFACTS_DIR / 'controls' / 'negative_controls.pkl',
        DATA_DIR / 'fcg.txt': DATA_DIR / 'fcg.txt',
    }
    if name in special:
        return special[name]
    matches = [p for p in PROJECT_ROOT.rglob(name) if '.ipynb_checkpoints' not in p.parts and '.git' not in p.parts]
    if len(matches) == 1:
        return matches[0]
    if not candidate.suffix and (candidate.is_absolute() or ':' in text):
        return PROJECT_ROOT
    return candidate

from pdm_learn.preprocessing import build_density_map, density_centers, densitymap, drop_nan, extract, mut_trim, normalize, trim, trim_pairs
from pdm_learn.modeling import KFold_PR, LOOCV, LOOCV_grouped_plot, area_table, core_predict, heatmap, importance_test, ks_pvalue
from pdm_learn.simulation import eps, partition


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import integrate as scipy_int
from sympy import *

In [ ]:
# Shared helper functions now live in src/pdm_learn.
# See the project setup cell at the top of this notebook for imports.


In [ ]:
# Shared helper functions now live in src/pdm_learn.
# See the project setup cell at the top of this notebook for imports.


In [ ]:
u = Symbol('u')
v = sqrt(1-u**2)
arr_x = partition(u, v, bounds=(-1,1), num=500).astype(float)
arr_y = np.array([v.subs(u, i).evalf() for i in arr_x]).astype(float)
arr_x /= np.std(arr_x)
arr_y /= np.std(arr_y)

In [ ]:
base_func = pd.DataFrame()
u = Symbol('u')

v = u
arr_x = partition(u, v, bounds=(-1,1), num=500).astype(float)
arr_y = np.array([v.subs(u, i).evalf() for i in arr_x]).astype(float)
arr_x /= np.std(arr_x)
arr_y /= np.std(arr_y)
arr_x -= np.mean(arr_x)
arr_y -= np.mean(arr_y)
base_func = pd.concat([base_func, pd.DataFrame(arr_x).T, pd.DataFrame(arr_y).T], axis=0)

v = u**2
arr_x = partition(u, v, bounds=(-1,1), num=500).astype(float)
arr_y = np.array([v.subs(u, i).evalf() for i in arr_x]).astype(float)
arr_x /= np.std(arr_x)
arr_y /= np.std(arr_y)
arr_x -= np.mean(arr_x)
arr_y -= np.mean(arr_y)
base_func = pd.concat([base_func, pd.DataFrame(arr_x).T, pd.DataFrame(arr_y).T], axis=0)

v = sin(u)
arr_x = partition(u, v, bounds=(-pi,pi), num=500).astype(float)
arr_y = np.array([v.subs(u, i).evalf() for i in arr_x]).astype(float)
arr_x /= np.std(arr_x)
arr_y /= np.std(arr_y)
arr_x -= np.mean(arr_x)
arr_y -= np.mean(arr_y)
base_func = pd.concat([base_func, pd.DataFrame(arr_x).T, pd.DataFrame(arr_y).T], axis=0)

v = Integer(0)
arr_x = partition(u, v, bounds=(0,1), num=251).astype(float)
arr_y = np.array([v.subs(u, i).evalf() for i in arr_x]).astype(float)
temp = arr_x.copy()
arr_x = np.append(arr_x, arr_y[1:-1])
arr_y = np.append(arr_y, temp[1:-1])
arr_x /= np.std(arr_x)
arr_y /= np.std(arr_y)
arr_x -= np.mean(arr_x)
arr_y -= np.mean(arr_y)
base_func = pd.concat([base_func, pd.DataFrame(arr_x).T, pd.DataFrame(arr_y).T], axis=0)

base_func

In [ ]:
# visualize graphs
eps_std = 0.5

for i in range(0, len(base_func), 2):
    x = base_func[i:i+1]
    y = base_func[i+1:i+2]
    x = x + eps(np.size(x), eps_std)
    y = y + eps(np.size(y), eps_std)
    plt.subplot(2, 3, int(i/2+1))
    plt.scatter(x, y, marker='.')

plt.suptitle(f'epsilon std: {eps_std}')
plt.show()

In [ ]:
# make simulated data sets
eps_std = 0
out = pd.DataFrame()

for i in range(0, len(base_func), 2):
    x = base_func[i:i+1].to_numpy()
    y = base_func[i+1:i+2].to_numpy()
    for _ in range(25):
        temp_x = x + eps(np.size(x), eps_std)
        temp_y = y + eps(np.size(y), eps_std)

        # normalize
        temp_x /= np.std(temp_x)
        temp_x -= np.mean(temp_x)
        temp_y /= np.std(temp_y)
        temp_y -= np.mean(temp_y)
        
        out = pd.concat([out, pd.DataFrame(temp_x), pd.DataFrame(temp_y)], axis=0)

out = out.reset_index(drop=True)
out

In [ ]:
out.to_csv(DATA_DIR / 'simulated' / 'positive.csv', index=False)

In [ ]:
a = 198

plt.scatter(out[a:a+1], out[a+1:a+2])

In [ ]:
eps_std = 0.5
x = arr_x + eps(np.size(arr_x), eps_std)
y = arr_y + eps(np.size(arr_y), eps_std)

plt.scatter(arr_x, arr_y, marker='.')
plt.title('Base')
plt.show()

plt.scatter(x, y, marker='.')
plt.title('Positive')
plt.show()

np.random.default_rng().shuffle(y)
plt.scatter(x, y, marker='.')
plt.title('Negative')
plt.show()